## 라쿠텐 top10 추출및 모든 리뷰데이터 2년치 추출코드

In [6]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import requests
import time
import random
import json
from datetime import datetime, timedelta

# ── 설정 ──────────────────────────────────────────────────────────
TARGET_URL       = "https://ranking.rakuten.co.jp/weekly/100944/p=1/"
RANK_SAVE_FILE   = "rakuten_rankings_current.jsonl"
REVIEW_SAVE_FILE = "rakuten_reviews_master.jsonl"

# 현재 날짜 기준 정확히 2년 전 날짜 계산
CUTOFF_DATE = datetime.now() - timedelta(days=365 * 2)

RAW_COOKIE = r"_ra=1773145336745|9810936d-5cda-4630-8e49-c98379684815; Rp=dc4c4487b0d3c4dce21f7d6bdbf70329f03486c8; rcxGlobal=d45a9a4c-4afa-4759-a9ed-229577264fb8; bm_mi=02819FD1874A677A9FC3219D127ECD18~YAAQB9ojF3ykzaGcAQAAMAK01x85bJZA06vzzbG4DIKAXg7mopEgiKN5lzrV/0AErGfSM50oZ5QfIMqbrcROt7DYSkrYw58Cz7vwEO2XkjKMkHGsBRiolwhSHY6BjhqOg8UrHpjdOwbKwGHWbgXrhYVrXbpqnuYxOnS6Rd4tbRsuPIrmZrskES38smu59dSiBK0bGYfeffgKl/2sz7WT7sEbAOAom3GeHhE5vSnLp8Unxda00YTc54V1R0SccenoAvO/8tAjlqf4fcZMyVfgWjmkkXpBvobpj5hyrWagxWfFFgn06n/L/mwrI22pddxKke0Bln2zVz7w9QvyrImNfBfU7zu1YGnnS6HKKDs9fU21qukz3/qjiTlpM3yQIgMrYmSo2aL/dooM6uWJl/I7WeDfGzeHGvHpBy6zNQ==~1; bm_sv=F3073B8724961BB7DCBE5A9516653181~YAAQbIj+eWOAwsucAQAA7Di01x/kkLOsyzmShVkJPcJf1O0TkzKdl7ZL7qgaen5E33IuCROVCPlrVdQr767lxL25yZRnHKhQTLV74swQf/WGDo3WVE2ehk0W7uv6G6PUyNE2N5dQyBl/0tnQF1TIL1GoIY1QVyysRmPhNLu9lndOUVbr8CgHlwjK+8cZfXcWG/rrLNFiEsHCtxhiQUijgpAP4TGE1raLDQ8pAaWOcclxAHnAjyBcQ2TkH+CwcSLNeCnH~1; ak_bmsc=6DA2C442401E92959215D4FC32333198~000000000000000000000000000000~YAAQB9ojF5AVzqGcAQAA0cu01x9fj3L5e+d9RAJ1hoLlH3K8NWtbYVM99UU1cPhBjs0l+RT4L93zSS2KCBZ2cF79c9IkAVFdcgGHbxP50Uf8nlO5I6u+fsByVK6zS4EhUNSCSizuhMPBdf0lHio/+lSN+8w0fTRgZfWwPNbWNRlfyg8SAriAihvo7Qf2txj9kJZGnaHPqgkCZXs11UwhehZ5Ak3MzYtTcojSc91bzqp+zzmmEbyhuYzqW5FWGr6d5k6LF157rv/BKCBjyW2dtk7TbRgJKgQGZz/gPYD8n8mDMroN36ptPxuByW5GOOwTwoCs8tDnUOjgOtm8M7mmdRNKlpJsUSJoOylT7fkQoeVPZkHsLG1j5QOPG60+P7zXsIeB4pddGO/PKsDF8HPAgLpOVb2Dgqz56kyCR4vfPNXZLawqlDXl8+8HXh2AL5KJmQIZMy0XvT/jsyXGkSBk6KO8l/jsyQp1AG570i/fWulONUZGzkQKF86JqOOaoDS7dUrDgp1mqvDR4lwtHSs=; krt_rewrite_uid=f8b15f33-cc3e-45c0-b1be-9a29a4cff1d2; Re=31.1.5.0.0.216348.3-31.1.5.0.0.216348.3; rat_v=05252dcb01d4e6863512b1a3e769b00e711919a"
SAFE_COOKIE = RAW_COOKIE.encode('utf-8').decode('latin-1', 'ignore')

REVIEW_API_URL = "https://web-gateway.rakuten.co.jp/review/itemshopreviewlist/get/v1"
REVIEW_HEADERS = {
    "authkey"     : "isrlPcMjUuXCVBUTVh91ZcHEfoI45CmPR",
    "content-type": "application/json; charset=UTF-8",
    "accept"      : "application/json, text/plain, */*",
    "user-agent"  : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    "cookie"      : SAFE_COOKIE,
    "referer"     : "https://review.rakuten.co.jp/",
}

MAX_PAGES = 100

# ── 함수: 별점 필터별 리뷰 수집 (날짜 제한 포함) ─────────────────────
def get_reviews_by_rating(shop_id, item_id, rating_filter):
    all_reviews = []
    page = 1
    has_next = True
    stop_collecting = False

    while has_next and page <= MAX_PAGES and not stop_collecting:
        payload = {
            "common": { "params" : {"device": "pc"}, "include": ["itemReviewList"] },
            "features": {
                "itemReviewList": {
                    "params": {
                        "shopId"             : int(shop_id),
                        "itemId"             : int(item_id),
                        "sort"               : "6", # 최신순 정렬
                        "page"               : str(page),
                        "hits"               : 30,
                        "filter"             : { "rating": rating_filter },
                        "includePickupReview": True,
                    }
                }
            },
        }
        try:
            resp = requests.post(REVIEW_API_URL, headers=REVIEW_HEADERS, json=payload, timeout=15)
            if resp.status_code not in [200, 207]: break

            data = resp.json()
            res_data = data.get("body", {}).get("itemReviewList", {}).get("data", {})
            reviews = res_data.get("reviews", [])

            if not reviews: break

            for rev in reviews:
                post_date_raw = rev.get("postDate") # "2026/03/14 10:00:00" 형태 대응
                if post_date_raw:
                    # 슬래시(/)나 하이픈(-) 모두 대응하기 위해 공백으로 자른 뒤 처리
                    date_str = post_date_raw.split(' ')[0]
                    # 라쿠텐 API에 따라 형식이 다를 수 있으므로 예외 처리 강화
                    try:
                        if '/' in date_str:
                            post_date = datetime.strptime(date_str, "%Y/%m/%d")
                        else:
                            post_date = datetime.strptime(date_str, "%Y-%m-%d")
                    except Exception as e:
                        print(f"날짜 변환 실패: {date_str} / {e}")
                        continue

                    # 날짜 비교: 기준일(2년 전)보다 과거이면 루프 중단
                    if post_date < CUTOFF_DATE:
                        stop_collecting = True
                        break

                all_reviews.append({
                    "shop_id"      : shop_id,
                    "item_id"      : item_id,
                    "rating_filter": rating_filter or "all",
                    "Nickname"     : rev.get("nickname"),
                    "Rating"       : rev.get("rating"),
                    "Body"         : rev.get("body"),
                    "PostDate"     : post_date_raw,
                    "Age"          : f"{rev.get('ageRange', '')}{rev.get('ageSuffix', '')}",
                    "Sex"          : rev.get("sex"),
                    "Sku"          : rev.get("skuInfo"),
                })

            if stop_collecting: break
            has_next = res_data.get("hasNextPage", False)
            page += 1
            time.sleep(1.8)

        except Exception as e:
            print(f"\n    ❌ 에러: {e}")
            break

    return all_reviews

# ── 함수: 별점 전략 실행 ──────────────────────────────────────────
def get_rakuten_all_reviews(shop_id, item_id, product_name):
    all_reviews = []
    print(f"\n  🚀 [{product_name[:35]}] 리뷰 수집 ({CUTOFF_DATE.strftime('%Y-%m-%d')} 이후)")

    for star in ["1", "2", "3", "4", "5"]:
        print(f"    ⭐{star}점 분석 중...", end="\r")
        reviews = get_reviews_by_rating(shop_id, item_id, star)
        all_reviews.extend(reviews)
        print(f"    ⭐{star}점: {len(reviews)}개 확보 완료")
        time.sleep(1.0)

    return all_reviews

# ── 메인 함수 ────────────────────────────────────────────────────
def fetch_rakuten_data():
    options = uc.ChromeOptions()
    options.add_argument("--window-size=1920,1080")
    driver = uc.Chrome(options=options)

    try:
        print(f"📡 라쿠텐 접속 중 (기준일: {CUTOFF_DATE.strftime('%Y-%m-%d')})")
        driver.get(TARGET_URL)
        time.sleep(random.uniform(7, 10))

        soup = BeautifulSoup(driver.page_source, "html.parser")
        all_items = soup.select("div.rnkRanking_top3box, div.rnkRanking_after4box")
        
        rank_data_list = []
        all_review_master = []
        rank_count = 1

        for item in all_items:
            if rank_count > 10: break
            try:
                review_link = item.select_one("a[href*='review.rakuten.co.jp/item']")
                shop_id = item_id = ""
                if review_link:
                    href = review_link.get("href", "")
                    parts = href.rstrip("/").split("/")
                    id_part = next((p for p in reversed(parts) if "_" in p), "")
                    if id_part:
                        shop_id, item_id = id_part.split("_", 1)

                if not shop_id or not item_id:
                    continue

                title_tag = item.select_one(".rnkRanking_itemName a")
                title = title_tag.get_text(strip=True) if title_tag else "N/A"
                shop_name = item.select_one(".rnkRanking_shop a").get_text(strip=True) if item.select_one(".rnkRanking_shop a") else "N/A"
                price = item.select_one(".rnkRanking_price").get_text(strip=True) if item.select_one(".rnkRanking_price") else "N/A"

                rank_data_list.append({
                    "rank": rank_count, "title": title, "shop_name": shop_name,
                    "price": price, "url": title_tag.get("href") if title_tag else "",
                    "shop_id": shop_id, "item_id": item_id, "platform": "Rakuten",
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                })

                reviews = get_rakuten_all_reviews(shop_id, item_id, title)
                all_review_master.extend(reviews)

                rank_count += 1
            except Exception as e:
                print(f"⚠️ {rank_count}위 파싱 오류: {e}")
                rank_count += 1
                continue

        with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
            for entry in rank_data_list: f.write(json.dumps(entry, ensure_ascii=False) + "\n")

        with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
            for rev in all_review_master: f.write(json.dumps(rev, ensure_ascii=False) + "\n")

        print(f"\n📊 작업 완료: 상품 {len(rank_data_list)}개 / 저장된 리뷰 {len(all_review_master)}개")

    finally:
        driver.quit()

if __name__ == "__main__":
    fetch_rakuten_data()

📡 라쿠텐 접속 중 (기준일: 2024-03-20)

  🚀 [総合ランキング1位【VT 大容量 パック 2種セット】【 スージング ] 리뷰 수집 (2024-03-20 이후)
    ⭐1점: 84개 확보 완료
    ⭐2점: 56개 확보 완료
    ⭐3점: 157개 확보 완료
    ⭐4점: 781개 확보 완료
    ⭐5점: 3000개 확보 완료

  🚀 [スキンクリア クレンズ オイル エコパック* (限定1種/通常4種) ] 리뷰 수집 (2024-03-20 이후)
    ⭐1점: 30개 확보 완료
    ⭐2점: 50개 확보 완료
    ⭐3점: 347개 확보 완료
    ⭐4점: 3000개 확보 완료
    ⭐5점: 3000개 확보 완료

  🚀 [【公式】Yunth 生ビタミンC 美白美容液 1ml×28包 | 美容] 리뷰 수집 (2024-03-20 이후)
    ⭐1점: 111개 확보 완료
    ⭐2점: 70개 확보 완료
    ⭐3점: 367개 확보 완료
    ⭐4점: 3000개 확보 완료
    ⭐5점: 3000개 확보 완료

  🚀 [スキンクリア クレンズ オイル (レギュラーボトル) (4種) 【アテ] 리뷰 수집 (2024-03-20 이후)
    ⭐1점: 27개 확보 완료
    ⭐2점: 58개 확보 완료
    ⭐3점: 361개 확보 완료
    ⭐4점: 3000개 확보 완료
    ⭐5점: 3000개 확보 완료

  🚀 [＼やよいWEEKで最大10%OFF+全品Pアップ／ポイント10倍!【資] 리뷰 수집 (2024-03-20 이후)
    ⭐1점: 5개 확보 완료
    ⭐2점: 14개 확보 완료
    ⭐3점: 36개 확보 완료
    ⭐4점: 454개 확보 완료
    ⭐5점: 1453개 확보 완료

  🚀 [乳液 セラミド 楽天ベストコスメ2023 殿堂入り 高保湿 さらさら ] 리뷰 수집 (2024-03-20 이후)
    ⭐1점: 16개 확보 완료
    ⭐2점: 24개 확보 완료
    ⭐3점: 47개 확보 완료
    ⭐4점: 186개 확보 완료
    ⭐5점

# 라쿠텐 번역 코드

In [ ]:
from deep_translator import GoogleTranslator
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import json, time, re
from tqdm import tqdm

# ── 설정 (환경에 맞춰 수정) ──────────────────────────────────────────
INPUT_FILE  = './rakuten_reviews_master.jsonl'
OUTPUT_FILE = 'rakuten_master_translated_jp_ko.jsonl'
BODY_COL    = 'Body'
N_SAMPLE    = None  # 전체 번역 시 None 유지

# [병렬 처리 핵심 설정]
CHUNK_SIZE  = 5      # 한 번에 묶어서 번역할 개수 (리뷰가 길면 3 권장)
MAX_WORKERS = 4      # 동시에 돌릴 스레드 수 (IP 차단 위험 시 2로 낮춤)
MAX_RETRIES = 3      # 실패 시 재시도 횟수
RETRY_SLEEP = 2.0    # 재시도 전 대기 시간
CHUNK_DELAY = 0.5    # ja->en 후 en->ko 넘어가기 전 짧은 휴식

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1. 조립 및 파싱 로직 (번호 매기기 방식)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_numbered(texts: list) -> str:
    """리뷰 리스트를 [1] 문장1 \n [2] 문장2 형태로 결합"""
    return "\n".join(f"[{i+1}] {str(t).strip()}" for i, t in enumerate(texts))

def parse_numbered(text: str, expected_n: int) -> list:
    """번역된 텍스트에서 [1], [2] 패턴을 찾아 리스트로 분리"""
    pattern = re.compile(r'\[(\d+)\]\s*(.*?)(?=\[\d+\]|$)', re.DOTALL)
    found = pattern.findall(text)
    result = {int(idx): body.strip() for idx, body in found}
    return [result.get(i + 1, "") for i in range(expected_n)]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2. 번역 엔진 (단건 및 청크)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def translate_single(text: str, src: str, tgt: str) -> str:
    """청크 번역 실패 시 개별적으로 번역하는 백업 함수"""
    if not text or not str(text).strip(): return ""
    for attempt in range(MAX_RETRIES):
        try:
            res = GoogleTranslator(source=src, target=tgt).translate(text)
            if res: return res.strip()
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
    return "번역실패"

def translate_chunk_logic(texts: list, src: str, tgt: str) -> list:
    """실질적인 청크 번역 수행 엔진"""
    if not texts: return []
    joined = build_numbered(texts)
    
    for attempt in range(MAX_RETRIES):
        try:
            result = GoogleTranslator(source=src, target=tgt).translate(joined)
            if not result: raise ValueError("응답 없음")
            
            parts = parse_numbered(result, len(texts))
            # 모든 파트가 정상 파싱되었는지 확인
            if all(p.strip() for p in parts):
                return parts
            
            # 파싱 실패한 항목만 단건 번역으로 보정
            for i, p in enumerate(parts):
                if not p: parts[i] = translate_single(texts[i], src, tgt)
            return parts
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
            
    return [translate_single(t, src, tgt) for t in texts]

def translate_chunk_2step(texts: list):
    """ja -> en -> ko 2단계 통합 실행 (병렬 작업 단위)"""
    # 1단계: 일 -> 영
    en_texts = translate_chunk_logic(texts, 'ja', 'en')
    time.sleep(CHUNK_DELAY)
    # 2단계: 영 -> 한
    ko_texts = translate_chunk_logic(en_texts, 'en', 'ko')
    return en_texts, ko_texts

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3. 메인 파이프라인 (tqdm 진행 상황 표시)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def main():
    print(f"📥 데이터 로드 중: {INPUT_FILE}")
    raw_records = []
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    raw_records.append(json.loads(line))
    except FileNotFoundError:
        print("❌ 파일을 찾을 수 없습니다.")
        return

    df = pd.DataFrame(raw_records)
    if N_SAMPLE: df = df.head(N_SAMPLE)
    
    total_count = len(df)
    print(f"✅ 총 {total_count:,}건 로드 완료 (컬럼: {list(df.columns)})")

    # 청크 분할
    texts = df[BODY_COL].fillna("").tolist()
    chunks = [texts[i:i+CHUNK_SIZE] for i in range(0, total_count, CHUNK_SIZE)]
    n_chunks = len(chunks)

    print(f"\n🚀 병렬 번역 시작 (스레드={MAX_WORKERS}, 청크={CHUNK_SIZE})")
    start_time = time.time()

    # ThreadPoolExecutor와 tqdm의 결합
    all_en, all_ko = [], []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # 진행상황을 chunk 단위로 표시
        chunk_results = list(tqdm(
            executor.map(translate_chunk_2step, chunks),
            total=n_chunks,
            desc="번역 진행률",
            unit="chunk"
        ))

    # 결과 취합
    for en_chunk, ko_chunk in chunk_results:
        all_en.extend(en_chunk)
        all_ko.extend(ko_chunk)

    df['Body_en'] = all_en[:total_count]
    df['Body_ko'] = all_ko[:total_count]

    elapsed = time.time() - start_time
    
    # 📊 간단 통계 및 저장
    success_rate = (df['Body_ko'] != "번역실패").mean() * 100
    print(f"\n⏱ 번역 완료: {elapsed/60:.1f}분 소요 (성공률: {success_rate:.1f}%)")

    print(f"💾 결과 저장 중: {OUTPUT_FILE}")
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for record in df.to_dict(orient='records'):
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    print("✨ 모든 작업이 끝났습니다!")

if __name__ == "__main__":
    main()

📥 데이터 로드 중: ./rakuten_reviews_master.jsonl
✅ 총 37,139건 로드 완료 (컬럼: ['shop_id', 'item_id', 'rating_filter', 'Nickname', 'Rating', 'Body', 'PostDate', 'Age', 'Sex', 'Sku'])

🚀 병렬 번역 시작 (스레드=4, 청크=5)


번역 진행률: 100%|██████████| 7428/7428 [1:29:17<00:00,  1.39chunk/s]  



⏱ 번역 완료: 89.3분 소요 (성공률: 100.0%)
💾 결과 저장 중: rakuten_master_translated_jp_ko.jsonl
✨ 모든 작업이 끝났습니다!


## 라쿠텐 키워드 kebert 1차 분류

In [ ]:
import re
import json
import pandas as pd
from tqdm import tqdm
from keybert import KeyBERT

# =========================
# 설정
# =========================
INPUT_FILE   = "./rakuten_master_translated_jp_ko.jsonl"
OUTPUT_CSV   = "./rakuten_keybert_en_categorized.csv"
OUTPUT_JSONL = "./rakuten_keybert_en_categorized.jsonl"

TEXT_COL       = "Body_en"
TOP_N_KEYWORDS = 5

CATEGORY_KEYWORDS = {
    "효과_성분": [
        "moistur", "hydrat", "hydrating", "whitening", "brighten", "elastic", "firm",
        "wrinkle", "anti-aging", "pore", "glow", "radian", "regenerat",
        "sooth", "calm", "antioxidant", "absorb", "penetrat", "tone", "improve",
        "vitamin", "retinol", "hyaluronic", "ceramide", "niacinamide", "peptid", "vegan",
        "plump", "clear", "even", "spot", "pigment", "aha", "bha", "acid", "exfoliat",
        # 추가: 피부타입 맥락
        "oily skin", "combination skin", "sensitive skin", "works for my skin",
        "dry skin type", "acne-prone skin",
        # 추가: 결과/시간 표현
        "result", "noticeabl", "overnight", "immediately", "after using",
        "after one week", "after a month", "after two week",
        # 추가: 사용 맥락
        "routine", "layer", "morning", "night cream",
        # 추가: 선케어 (제품군 해당 시)
        "spf", "sunscreen", "uv", "sun protect", "reef safe",
    ],
    "사용감_텍스처": [
        "appl", "blend", "texture", "consistenc", "watery", "runny", "thick", "viscos",
        "stick", "tacky", "fresh", "light", "weightless", "heavy", "soft", "smooth",
        "stiff", "greasy", "pill", "flake", "feel", "finish", "rub",
        "oily", "matte", "dewy", "sink", "patchy", "chalky", "white cast",
        # 추가: 메이크업 베이스/선케어 관련
        "pore-filling", "pore filling", "blur", "setting", "blot", "primer",
        "spread", "glide", "pack", "apply thin", "build up",
    ],
    "향_냄새": [
        "scent", "smell", "fragranc", "unscented", "fragrance-free", "odor",
        "subtle", "mild", "strong", "overpowering", "artificial", "natural", "perfume",
        "stink", "aroma", "nose",
        # 추가
        "whiff", "chemical smell", "medicin", "floral", "citrus",
    ],
    "피부_트러블_부작용": [
        "trouble", "breakout", "pimple", "acne", "irritat", "sting", "burn", "itch",
        "red", "redness", "peel", "tight", "sensitiv", "allerg", "dermatitis",
        "reaction", "side effect", "break out", "rash", "harsh",
        "drying", "dried out", "flaky", "dry patch",
        "clog", "purg", "cyst", "bump",
        # 추가: 자극 표현
        "tingle", "sting", "inflam", "swell", "hive", "welt",
        "made my skin worse", "broke me out", "not agree",
    ],
    "포장_배송": [
        "packag", "box", "bottle", "container", "case", "pump", "tube", "ship",
        "deliver", "late", "slow", "arriv", "damag", "broken",
        "leak", "spill", "wrap",
        "fast ship", "arrived fast", "quick deliver",
        "dropper", "cap", "lid", "spray", "nozzle", "shipped", "unseal",
        # 추가
        "packaging", "travel size", "full size", "well-packaged", "poorly packaged",
        "dent", "crush", "tamper",
    ],
    "가격_가성비": [
        "price", "cost", "valu", "expensiv", "pricy", "cheap", "afford", "reasonabl",
        "sale", "discount", "coupon", "buck", "money", "worth", "deal", "size", "amount",
        "pricey", "bargain", "rip off", "waste",
        # 추가: 용량 관련 가성비 표현
        "goes a long way", "a little goes", "last a long time", "last me",
        "small amount", "tiny bit", "lasts forever", "run out fast", "finish quickly",
    ],
    "고객서비스": [
        "custom", "service", "support", "respond", "response", "refund", "return",
        "exchang", "complain", "inquir", "answer", "contact", "issue",
        # 추가
        "seller", "vendor", "representative", "chat", "email them", "called",
        "waited", "resolve", "compensat",
    ],
    "제품불량": [
        "defect", "defective", "faulty", "bug", "dirt", "contaminat", "spoil",
        "weird", "fake", "counterfeit", "knockoff", "differ", "mold", "trash",
        "expir", "rancid", "separat", "empty",
        # 추가
        "smell off", "color off", "look different", "wrong product", "not what",
        "different from", "old stock", "bad batch",
    ],
    "재구매_추천": [
        "repurchas", "buy again", "reorder", "recommend", "holy grail", "staple",
        "go-to", "favorit", "gift", "friend", "keep us", "definitely",
        "love it",
        "hg", "restock", "10/10", "must have", "obsessed",
        # 추가: 비교/전환 표현
        "better than", "switch from", "switch to", "compared to", "used to use",
        "replace", "converted",
        # 추가: 일반 강한 긍정 (내용 없는 짧은 리뷰 흡수)
        "highly recommend", "great product", "works well", "works great",
        "amazing product", "excellent", "perfect product", "love this",
        "love how", "love that", "so good", "so happy",
    ],
    
     "부정_리뷰": [
        # 추가: 부정 추천 표현
        "disappoint", "not recommend", "waste of money", "regret buying",
    ],

    "커버력_색상": [
        "cover", "coverage", "color", "shade", "tone", "tint", "pigment",
        "bright", "dark", "ashy", "oxidiz", "orang", "yellow", "match", "pale",
        "undertone", "fair", "sheer", "opaque", "swatch",
        # 추가
        "full coverage", "medium coverage", "buildable", "natural finish",
        "foundation", "concealer", "bb cream", "cc cream", "tinted",
        "skin tone", "complexion", "too light", "too dark", "perfect match",
    ],
    "지속력_밀착력": [
        "last", "lasting", "longevity", "stay", "adher", "crease", "melt", "fade",
        "long-lasting", "all day", "wear", "hold", "slip", "smudg", "transfer",
        "rub off", "budge", "separate",
        # 추가: 환경 내구성
        "sweat", "sweatproof", "waterproof", "water resistant", "humid",
        "through the day", "by noon", "by midday", "hours later",
        "8 hour", "12 hour", "24 hour",
    ],
}

# =========================
# 유틸
# =========================
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def clean_light(text):
    text = str(text).replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip()

def extract_keywords(text, top_n=5):
    text = str(text).strip()
    if not text:
        return []
    try:
        kws = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),
            stop_words="english",
            top_n=top_n,
            use_mmr=True,
            diversity=0.5
        )
        return [kw for kw, _ in kws]
    except Exception:
        return []

def rule_classify(keywords: list, text: str) -> tuple:
    combined = " ".join(keywords).lower() + " " + text.lower()
    scores = {}
    for cat, kw_list in CATEGORY_KEYWORDS.items():
        count = sum(1 for kw in kw_list if kw in combined)
        if count > 0:
            scores[cat] = count

    if not scores:
        return "unclassified", ["unclassified"]

    sorted_cats = sorted(scores, key=scores.get, reverse=True)
    return sorted_cats[0], sorted_cats[:3]

# =========================
# 모델 로드
# =========================
kw_model = KeyBERT("all-MiniLM-L6-v2")

# =========================
# 데이터 로드
# =========================
df = load_jsonl(INPUT_FILE)
df = df.dropna(subset=[TEXT_COL]).copy()
df = df[df[TEXT_COL].astype(str).str.strip() != ""].reset_index(drop=True)
df["text_for_model"] = df[TEXT_COL].apply(clean_light)
print(f"유효 데이터: {len(df):,}건")

# =========================
# KeyBERT 키워드 추출 + 규칙 분류
# =========================
keywords_list      = []
primary_categories = []
categories_list    = []

for text in tqdm(df["text_for_model"], desc="KeyBERT 분류 (EN)"):
    kws           = extract_keywords(text, top_n=TOP_N_KEYWORDS)
    primary, cats = rule_classify(kws, text)
    keywords_list.append(kws)
    primary_categories.append(primary)
    categories_list.append(cats)

df["keybert_keywords"]  = keywords_list
df["primary_category"]  = primary_categories
df["categories"]        = categories_list

# =========================
# 결과 출력
# =========================
total        = len(df)
classified   = (df["primary_category"] != "unclassified").sum()
unclassified = (df["primary_category"] == "unclassified").sum()

print(f"\n분류 완료: {classified:,}건 ({classified/total*100:.1f}%)")
print(f"미분류:    {unclassified:,}건 ({unclassified/total*100:.1f}%)")
print("\n카테고리 분포:")
print(df["primary_category"].value_counts())

# =========================
# 저장
# =========================
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

print(f"\nCSV:   {OUTPUT_CSV}")
print(f"JSONL: {OUTPUT_JSONL}")

print("\n샘플:")
print(df[[TEXT_COL, "keybert_keywords", "primary_category", "categories"]].head(10))

## 라쿠텐 키워드 gpt 2차 분류

In [ ]:
import json
import re
from openai import OpenAI

# ── 설정 ──────────────────────────────────────────────────────────────
INPUT_JSONL  = "./rakuten_keybert_en_categorized.jsonl"  # KeyBERT 결과 JSONL
OUTPUT_JSONL = "./rakuten_final_categorized.jsonl"
BATCH_SIZE   = 30
TEXT_COL     = "Body_en"

CATEGORIES = [
    "효과_성분", "사용감_텍스처", "향_냄새", "피부_트러블_부작용","부정_리뷰",
    "포장_배송", "가격_가성비", "고객서비스", "제품불량",
    "재구매_추천", "커버력_색상", "지속력_밀착력", "미분류"
]

client = OpenAI()

# ── GPT 배치 분류 ──────────────────────────────────────────────────────
def gpt_classify_batch(batch: list[dict]) -> dict:
    """batch: [{"idx": i, "keywords": [...], "text": "..."}]
    반환: {idx: {"primary_category": ..., "categories": [...]}}
    """
    prompt_items = "\n".join(
        f"[{item['idx']}] keywords={item['keywords']} | text={item['text'][:300]}"
        for item in batch
    )
    system_msg = f"""뷰티 제품 리뷰를 아래 카테고리 중 하나로 분류하세요.
카테고리: {CATEGORIES}

각 리뷰에 대해 JSON 배열로 응답하세요:
[{{"idx": 번호, "primary_category": "카테고리명", "categories": ["카테고리1", ...]}}]
primary_category는 가장 핵심 카테고리 1개, categories는 해당되는 카테고리 모두."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt_items}
        ],
        temperature=0
    )

    raw = response.choices[0].message.content
    # JSON 배열 파싱
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    if not match:
        return {}
    results = json.loads(match.group())
    return {r["idx"]: r for r in results}

# ── 데이터 로드 ────────────────────────────────────────────────────────
records = []
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

classified   = [r for r in records if r.get("primary_category") != "unclassified"]
unclassified = [r for r in records if r.get("primary_category") == "unclassified"]

print(f"전체: {len(records)} | 분류완료: {len(classified)} | GPT 재분류 대상: {len(unclassified)}")

# ── GPT 배치 실행 ──────────────────────────────────────────────────────
idx_to_record = {i: rec for i, rec in enumerate(unclassified)}
batches = [
    [
        {
            "idx": i,
            "keywords": rec.get("keybert_keywords", []),
            "text": str(rec.get(TEXT_COL, ""))[:300]
        }
        for i, rec in list(idx_to_record.items())[start:start+BATCH_SIZE]
    ]
    for start in range(0, len(unclassified), BATCH_SIZE)
]

print(f"배치 수: {len(batches)} ({BATCH_SIZE}건씩)")

for b_idx, batch in enumerate(batches):
    results = gpt_classify_batch(batch)
    for item in batch:
        i = item["idx"]
        if i in results:
            idx_to_record[i]["primary_category"] = results[i]["primary_category"]
            idx_to_record[i]["categories"]       = results[i]["categories"]
        else:
            idx_to_record[i]["primary_category"] = "이분류"
            idx_to_record[i]["categories"]       = []
    print(f"  배치 {b_idx+1}/{len(batches)} 완료")

# ── 결과 저장 ──────────────────────────────────────────────────────────
final_records = classified + list(idx_to_record.values())

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in final_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"\n저장 완료: {OUTPUT_JSONL} ({len(final_records)}건)")

# ── 분류 결과 확인 ─────────────────────────────────────────────────────
from collections import Counter
cats = [r["primary_category"] for r in final_records]
for cat, cnt in Counter(cats).most_common():
    print(f"  {cat}: {cnt}")